# Agentic REPL wheel staging: download + publish

Downloads `llama_cpp_python-0.3.31-py3-none-manylinux_2_35_x86_64.whl` from GitHub Releases and publishes it as the Kaggle Dataset `ankitdash24/agentic-repl-llama-cpp-wheel`. CPU-only, internet-enabled, no GPU quota consumed.

## 1. Authenticate the Kaggle CLI from the staged token dataset

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
print('/kaggle/input contents:')
for entry in sorted(INPUT_ROOT.rglob('*')):
    print(' ', entry)

def find_input_file(dataset_slug, filename):
    flat = INPUT_ROOT / dataset_slug / filename
    if flat.is_file():
        return flat
    nested = sorted(INPUT_ROOT.glob(f'datasets/*/{dataset_slug}/{filename}'))
    return nested[0] if nested else None

CREDENTIALS_SRC = find_input_file('agentic-repl-kaggle-token', 'kaggle.json')
if CREDENTIALS_SRC is None:
    raise SystemExit('No kaggle.json found under /kaggle/input.')
print('Using Kaggle credentials from:', CREDENTIALS_SRC)

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(CREDENTIALS_SRC, KAGGLE_CONFIG_DIR / 'kaggle.json')
(KAGGLE_CONFIG_DIR / 'kaggle.json').chmod(0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print('kaggle CLI ready')


## 2. Download the wheel (resumable, size-verified)

In [ ]:
import time
import urllib.error
import urllib.request

WHEEL_URL = 'https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.31-cu125/llama_cpp_python-0.3.31-py3-none-manylinux_2_35_x86_64.whl'
WHEEL_FILE = 'llama_cpp_python-0.3.31-py3-none-manylinux_2_35_x86_64.whl'
EXPECTED_SIZE_BYTES = 1832490513
WHEEL_DIR = Path('/kaggle/tmp/wheel')
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
destination = WHEEL_DIR / WHEEL_FILE
CHUNK_SIZE = 8 * 1024 * 1024
MAX_ATTEMPTS = 10

started = time.perf_counter()
for attempt in range(1, MAX_ATTEMPTS + 1):
    existing = destination.stat().st_size if destination.exists() else 0
    headers = {'Range': f'bytes={existing}-'} if existing else {}
    request = urllib.request.Request(WHEEL_URL, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            mode = 'ab' if existing and response.status == 206 else 'wb'
            downloaded = existing if mode == 'ab' else 0
            print(f'attempt {attempt}: status={response.status} mode={mode} resume_from={downloaded}')
            with destination.open(mode) as out_file:
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    downloaded += len(chunk)
    except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as exc:
        print(f'attempt {attempt} failed: {exc!r} -- retrying')
        continue
    final_size = destination.stat().st_size
    print(f'attempt {attempt} ended with {final_size} bytes on disk')
    if final_size >= EXPECTED_SIZE_BYTES:
        break
else:
    raise SystemExit(f'Download incomplete after {MAX_ATTEMPTS} attempts.')

elapsed = time.perf_counter() - started
final_size = destination.stat().st_size
if final_size != EXPECTED_SIZE_BYTES:
    raise SystemExit(f'size {final_size} != expected {EXPECTED_SIZE_BYTES} -- aborting.')
print(f'Downloaded {destination} ({final_size/1e9:.2f} GB) in {elapsed:.0f}s '
      f'({(final_size/1e6)/elapsed:.1f} MB/s average)')


## 3. Verify it's a real, complete wheel (valid zip)

In [ ]:
import zipfile

with zipfile.ZipFile(destination) as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise SystemExit(f'Corrupt member in wheel zip: {bad_file}')
    names = archive.namelist()
print(f'Valid zip archive, {len(names)} entries, e.g.:', names[:5])


## 4. Download extra pure-Python dependencies not already on Kaggle

In [ ]:
EXTRA_PACKAGES = ['diskcache']
for package in EXTRA_PACKAGES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'download', package, '--no-deps', '-d', str(WHEEL_DIR)],
        capture_output=True, text=True, timeout=120,
    )
    print(f'--- pip download {package} ---')
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
    result.check_returncode()
print('files in WHEEL_DIR now:', sorted(p.name for p in WHEEL_DIR.glob('*.whl')))


## 5. Publish as a Kaggle Dataset

In [ ]:
import json as json_module

TARGET_DATASET_ID = 'ankitdash24/agentic-repl-llama-cpp-wheel'
TARGET_DATASET_TITLE = 'agentic-repl-llama-cpp-wheel'
metadata = {
    'title': TARGET_DATASET_TITLE,
    'id': TARGET_DATASET_ID,
    'licenses': [{'name': 'unknown'}],
}
(WHEEL_DIR / 'dataset-metadata.json').write_text(json_module.dumps(metadata, indent=2))

check = subprocess.run(['kaggle', 'datasets', 'status', TARGET_DATASET_ID], capture_output=True, text=True)
dataset_exists = check.returncode == 0
print('dataset_exists =', dataset_exists)
cmd = (
    ['kaggle', 'datasets', 'version', '-p', str(WHEEL_DIR), '-m', 'update']
    if dataset_exists else
    ['kaggle', 'datasets', 'create', '-p', str(WHEEL_DIR)]
)
result = subprocess.run(cmd, capture_output=True, text=True)
print('--- stdout ---')
print(result.stdout)
print('--- stderr ---')
print(result.stderr)
result.check_returncode()
print('Published dataset:', TARGET_DATASET_ID)
